# Module 2: Financial RAG System
Retrieval-Augmented Generation (RAG) allows an LLM to answer questions based on a specific set of documents, like an annual report or SEC filing, rather than relying on its internal memory.


In [ ]:
!pip install -q langchain langchain-google-genai chromadb python-dotenv


In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# Load environment variables
load_dotenv()


## 1. Configure the LLM
We will use Google's Gemini. Make sure you have `GOOGLE_API_KEY` in your `.env` file.


In [ ]:
if 'GOOGLE_API_KEY' not in os.environ:
    # Fallback to asking user directly if not in env
    import getpass
    os.environ['GOOGLE_API_KEY'] = getpass.getpass('Enter your Google API Key: ')

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model='models/embedding-001')


## 2. Load and Chunk Financial Documents
Let's load a sample financial article or report from the web.


In [ ]:
url = 'https://investor.apple.com/investor-relations/default.aspx' # Example URL
loader = WebBaseLoader(url)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
print(f'Split into {len(splits)} chunks.')


## 3. Create the Vector Database
We'll use ChromaDB to store our embeddings locally.


In [ ]:
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings, persist_directory='./chroma_db')
retriever = vectorstore.as_retriever()


## 4. Query the RAG System
Now we can ask specific financial questions.


In [ ]:
system_prompt = (
    "You are an expert financial analyst. Use the given context to answer the question. "
    "If you don't know the answer, say that you don't know.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

response = rag_chain.invoke({"input": "What are the key highlights for investors?"})
print(response['answer'])
